[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn.functional as F

In [39]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # pass  # extract patches, apply kernel, handle stride/padding
    x_padded = F.pad(x, (padding, padding, padding, padding))

    h_out = (x_padded.shape[2] - weight.shape[2]) // stride + 1
    w_out = (x_padded.shape[3] - weight.shape[3]) // stride + 1
    c_out = weight.shape[0]
    output = torch.zeros((x.shape[0], c_out, h_out, w_out), device=x.device)
    
    h_start = 0
    w_start = 0
    h_end = h_start + weight.shape[2]
    w_end = w_start + weight.shape[3]
    while True:
        patch = x_padded[:, :, h_start:h_end, w_start:w_end]
        # print(patch.shape, weight.shape)
        
        patch_flat = patch.reshape(patch.shape[0], -1)
        # patch_flat: (B, C_in * kH * kW)

        weight_flat = weight.reshape(weight.shape[0], -1)
        # weight_flat: (C_out, C_in * kH * kW)
        # print(patch_flat.shape, weight_flat.shape)
        output[:, :, h_start // stride, w_start // stride] = patch_flat @ weight_flat.T # per sample per channel, or use torch.mul() or * with loop
        
                
        w_start += stride
        w_end += stride
        if w_end > x_padded.shape[3]:
            w_start = 0
            w_end = weight.shape[3]
            h_start += stride
            h_end += stride
            if h_end > x_padded.shape[2]:
                break
    
    return output + (bias.view(1, -1, 1, 1) if bias is not None else 0)

In [40]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [41]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (0.8ms)
  ✅ [2/5] Matches F.conv2d (1.4ms)
  ✅ [3/5] With padding (0.7ms)
  ✅ [4/5] With stride (0.5ms)
  ✅ [5/5] Gradient flow (0.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (3.9ms total)
  Progress saved. Run status() to see your dashboard.

